# 学習したTTSモデルを使って音声を合成する
このファイルを`espnet/egs2/○○/tts1/`フォルダに置いて、上のブロックから順番に実行する

In [1]:
from espnet2.bin.tts_inference import Text2Speech
from espnet2.utils.types import str_or_none

モデルと設定ファイルのパスを記述する<br>
妙な処理をしていなければどちらのファイルも`exp/tts_train_○○/`にあるはず

In [35]:
# /home/{ユーザー名}/espnet2/egs2/{レシピ名}/tts1
%cd /home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1

import os

# 音声をwavファイルに出力するかどうか
save_wavfile = True
# 音声ファイルの保存にscipyかpysoundfile(sf)どちらを使うか
# save_method = "scipy"
save_method = "sf"
# モデルのパス
model = "/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/save/300epoch.pth"
# 学習時の設定ファイルのパス (指定しなければモデルのパスから勝手に探す)
config = "/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/config.yaml"
if config == None or config == '':
    config = os.path.join(os.path.dirname(model), "config.yaml")


# colabのサンプルに書いてあったもの
# @title Choose Japanese model { run: "auto" }
# lang = 'Japanese'
# tag = 'kan-bayashi/jsut_full_band_vits_prosody' #@param ["kan-bayashi/jsut_tacotron2", "kan-bayashi/jsut_transformer", "kan-bayashi/jsut_fastspeech", "kan-bayashi/jsut_fastspeech2", "kan-bayashi/jsut_conformer_fastspeech2", "kan-bayashi/jsut_conformer_fastspeech2_accent", "kan-bayashi/jsut_conformer_fastspeech2_accent_with_pause", "kan-bayashi/jsut_vits_accent_with_pause", "kan-bayashi/jsut_full_band_vits_accent_with_pause", "kan-bayashi/jsut_tacotron2_prosody", "kan-bayashi/jsut_transformer_prosody", "kan-bayashi/jsut_conformer_fastspeech2_tacotron2_prosody", "kan-bayashi/jsut_vits_prosody", "kan-bayashi/jsut_full_band_vits_prosody", "kan-bayashi/jvs_jvs010_vits_prosody", "kan-bayashi/tsukuyomi_full_band_vits_prosody"] {type:"string"}
# vocoder_tag = 'none' #@param ["none", "parallel_wavegan/jsut_parallel_wavegan.v1", "parallel_wavegan/jsut_multi_band_melgan.v2", "parallel_wavegan/jsut_style_melgan.v1", "parallel_wavegan/jsut_hifigan.v1"] {type:"string"}

/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1


In [36]:
text2speech = Text2Speech.from_pretrained(
    model_file=model,
    train_config=config,
    # web上のモデルをダウンロードするときの設定
    # model_tag=str_or_none(tag),
    # vocoder_tag=str_or_none(vocoder_tag),
    device="cpu", # "cuda"にしてもいいけどcpuで十分な速度  
    # Only for Tacotron 2 & Transformer
    threshold=0.5,
    # Only for Tacotron 2
    minlenratio=0.0,
    maxlenratio=10.0,
    use_att_constraint=False,
    backward_window=1,
    forward_window=3,
    # Only for FastSpeech & FastSpeech2 & VITS
    speed_control_alpha=1.0,
    # Only for VITS
    noise_scale=0.333,
    noise_scale_dur=0.333,
)

In [37]:
from IPython.display import Audio
import torch

if save_method == "sf":
    import soundfile as sf
elif save_method == "scipy":
    from scipy.io.wavfile import  write

# ここを書き換えてこのブロックを実行すれば音声を生成できる
text = "水をマレーシアから買わなくてはならないのです。"

# ここで推論
with torch.no_grad():
    wav = text2speech(text)["wav"]

# tensorからndarrayに変換
audio_array = wav.view(-1).cpu().numpy()

samplerate=text2speech.fs

# 音声をファイルに保存する
# カレントディレクトリにinferenceフォルダが作られるから注意
# 音声ファイルの名前は 発話の内容の頭から10文字 + .wav
if save_wavfile:
    # 保存するファイル名を
    os.makedirs("inference", exist_ok=True)
    if len(text) > 10:
        filename = text[:10]
    else:
        filename = text
    filename += ".wav"
    
    if save_method == "sf":
        sf.write(os.path.join("inference", filename), audio_array, samplerate)
    elif save_method == "scipy":
        write(os.path.join("inference", filename) , samplerate, audio_array)

# ノートブック上で音声を表示する
Audio(audio_array, rate=samplerate)